# 03 - Backtest Review

Loads backtest / experiment records written by the Rust CLI into
`data/experiments/` and shows their metrics side by side using
`guardrail_lab.experiments`.

**Offline-safe:** if no records exist the notebook prints a hint.


In [ ]:
# --- Guardrail Alpha notebook bootstrap (offline-safe) ---
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Return the first ancestor that contains python-lab/guardrail_lab."""
    for candidate in [start, *start.parents]:
        if (candidate / "python-lab" / "guardrail_lab").is_dir():
            return candidate
    return start


# In a notebook __file__ is undefined, so anchor on the current working dir.
REPO_ROOT = _find_repo_root(Path.cwd())
LAB_PATH = REPO_ROOT / "python-lab"
if str(LAB_PATH) not in sys.path:
    sys.path.insert(0, str(LAB_PATH))


def _first_existing(*candidates: Path):
    """Return the first path that exists, else None."""
    for path in candidates:
        if path.exists():
            return path
    return None


DATA_DIR = REPO_ROOT / "data"
CONFIG_DIR = REPO_ROOT / "configs"

# Prefer a real run; fall back to the seeded demo artifacts if present.
DB_PATH = _first_existing(
    DATA_DIR / "guardrail_alpha.db",
    DATA_DIR / "demo_guardrail_alpha.db",
)
RUN_REPORT_PATH = _first_existing(
    DATA_DIR / "run_report.json",
    DATA_DIR / "demo_run_report.json",
)
EXPERIMENTS_DIR = DATA_DIR / "experiments"
BACKTESTS_DIR = DATA_DIR / "backtests"
ELIGIBLE_ASSETS_PATH = CONFIG_DIR / "eligible_assets.bsc.json"
ASSET_CATEGORIES_PATH = CONFIG_DIR / "asset_categories.json"

NO_DATA_HINT = (
    "No data found under data/. Run the agent (paper/live) or seed a demo run "
    "first, e.g.  python3 -m guardrail_lab.seed  (from python-lab/), then "
    "re-run this notebook. The notebook is offline-safe and will not raise."
)

print("repo root :", REPO_ROOT)
print("event log :", DB_PATH if DB_PATH else "(none yet)")
print("run report:", RUN_REPORT_PATH if RUN_REPORT_PATH else "(none yet)")


In [ ]:
def render_table(rows, columns, title=None):
    """Print an aligned text table. rows: list[dict]; columns: list[(key,label)]."""
    if title:
        print(title)
        print("=" * len(title))
    if not rows:
        print("(no rows)")
        return
    labels = [label for _, label in columns]
    keys = [key for key, _ in columns]
    cells = [[("" if r.get(k) is None else str(r.get(k))) for k in keys] for r in rows]
    widths = [
        max(len(labels[i]), *(len(row[i]) for row in cells)) for i in range(len(keys))
    ]
    header = "  ".join(labels[i].ljust(widths[i]) for i in range(len(keys)))
    print(header)
    print("  ".join("-" * widths[i] for i in range(len(keys))))
    for row in cells:
        print("  ".join(row[i].ljust(widths[i]) for i in range(len(keys))))


## Load experiment / backtest records


In [ ]:
from guardrail_lab.experiments import load_experiments, compare_table

# Look in both the experiments dir and the backtests dir.
experiments = load_experiments(str(EXPERIMENTS_DIR))
backtests = load_experiments(str(BACKTESTS_DIR))
records = experiments + backtests

if not records:
    print(NO_DATA_HINT)
    print("\nLooked in:")
    print(" -", EXPERIMENTS_DIR)
    print(" -", BACKTESTS_DIR)
else:
    print(f"Loaded {len(records)} record(s) "
          f"({len(experiments)} experiments, {len(backtests)} backtests).")


## Metrics comparison


In [ ]:
def _fmt(value, suffix=""):
    if value is None:
        return "n/a"
    return f"{value:,.3f}{suffix}" if isinstance(value, float) else f"{value}{suffix}"

if records:
    rows = compare_table(records)
    table = [
        {
            "tag": r["tag"],
            "preset": r["preset"],
            "steps": r["steps"],
            "fng": r["fear_greed"],
            "return": _fmt(r["total_return_pct"], "%"),
            "max_dd": _fmt(r["max_drawdown_pct"], "%"),
            "trades": r["trade_count"],
            "win%": _fmt(r["win_rate_pct"], "%"),
            "pf": _fmt(r["profit_factor"]),
            "calmar": _fmt(r["calmar_ratio"]),
            "excess": _fmt(r["excess_return_pct"], "%"),
            "final_nav": _fmt(r["final_nav_usd"]),
        }
        for r in rows
    ]
    render_table(
        table,
        [
            ("tag", "TAG"),
            ("preset", "PRESET"),
            ("steps", "STEPS"),
            ("fng", "F&G"),
            ("return", "RETURN"),
            ("max_dd", "MAX_DD"),
            ("trades", "TRADES"),
            ("win%", "WIN"),
            ("pf", "PROFIT_FACTOR"),
            ("calmar", "CALMAR"),
            ("excess", "EXCESS"),
            ("final_nav", "FINAL_NAV"),
        ],
        title="Backtest / experiment comparison",
    )
else:
    print(NO_DATA_HINT)


## Best / worst by total return


In [ ]:
if records:
    scored = [
        r for r in compare_table(records) if r["total_return_pct"] is not None
    ]
    if scored:
        best = max(scored, key=lambda r: r["total_return_pct"])
        worst = min(scored, key=lambda r: r["total_return_pct"])
        print(f"Best total return : {best['tag']} "
              f"({best['total_return_pct']:.3f}%, preset={best['preset']})")
        print(f"Worst total return: {worst['tag']} "
              f"({worst['total_return_pct']:.3f}%, preset={worst['preset']})")
    else:
        print("No numeric total_return_pct values to rank.")
else:
    print(NO_DATA_HINT)


## Notes

* `EXCESS` is return over the buy-and-hold benchmark; a negative
  value means the strategy trailed simply holding the basket over
  that window.
* `CALMAR` = return / max drawdown; high values reflect very small
  drawdowns in short synthetic runs.
